In [ ]:
# !pip uninstall grpo -yes
# !pip install --upgrade trl


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
from dataclasses import dataclass, field

In [ ]:
def accuracy_reward(completions, **kwargs):
    # 实现准确度奖励逻辑
    return 0

def format_reward(completions, **kwargs):
    # 实现范式奖励逻辑
    return 0

reward_functions = [accuracy_reward, format_reward]


In [ ]:
from datasets import load_dataset
# 假设 make_conversation 函数的功能是将问题与提示信息组合
def make_conversation(example):
    # 这里的逻辑需要根据实际需求来编写
    # 示例逻辑：将 problem 字段内容添加到 prompt 中
    example['prompt'] = [example['problem']] if 'problem' in example else []
    return example

# 定义并加载数据集
def load_math_dataset():
    """Load and prepare the mathematics dataset."""
    dataset = load_dataset(
        "AI-MO/NuminaMath-TIR",
        name="default",
        split=['train', 'test']
    )
    
    # Convert splits into dictionary
    dataset = {
        'train': dataset[0],
        'test': dataset[1]
    }
    
    # Apply conversation format
    for split in dataset:
        dataset[split] = dataset[split].map(make_conversation)
        # Remove 'messages' column if exists
        if "messages" in dataset[split].column_names:
            dataset[split] = dataset[split].remove_columns("messages")
    
    return dataset

dataset = load_math_dataset()


In [ ]:
# 如果函数在当前文件中定义
def get_reward_functions(script_args):
    # 函数实现代码
    pass

In [ ]:
# 定义训练参数
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_steps=500,
)

In [ ]:

model = "Qwen/Qwen2.5-7B-Instruct"

# 将 TrainingArguments 对象转换为字典，并初始化 GRPOConfig 对象
grpo_config = GRPOConfig(
    **training_args.to_dict(), 
    **{ 
        # 这里移除了 model_init_kwargs
        # 由于我们直接传入了已实例化的 'model' 对象，
        # 所以 GRPOTrainer 不需要 model_init_kwargs 来初始化模型
    }
)

# 初始化 GRPOTrainer
grpo_trainer = GRPOTrainer(
    # 传入已初始化的 Qwen 模型
    model=model,                      
    # 传入之前步骤中获取的奖励函数列表
    reward_funcs=reward_functions,    
    # 传入由 TrainingArguments 创建的 GRPOConfig 对象
    args=grpo_config,                
    # 传入训练数据集
    train_dataset=dataset['train'],   
    # 传入评估数据集
    eval_dataset=dataset['test']       
)




In [ ]:
# 启动 GRPO 训练循环
# 调用 grpo_trainer 对象的 train 方法来开始训练过程
# train 方法会根据之前配置好的训练参数、奖励函数、数据集等进行训练
# 训练完成后，会将训练结果存储在 train_result 变量中，
# 这个结果通常包含了训练过程中的一些统计信息，比如损失值、训练步数等
train_result = grpo_trainer.train()